# Midterm Project: Empirical Analysis of "Shuffling" Transformation in Time Series Modeling

This Jupyter notebook provides a complete, interactive implementation of the midterm project. It investigates the impact of shuffling time series data (Bitcoin daily returns from 2016-2025) prior to ARIMA model estimation, focusing on three dimensions:

1. **Estimation Biasness**: How shuffling affects parameter estimates.
2. **Prediction Reasonableness**: Impact on out-of-sample forecast accuracy.
3. **Inference Validity**: Consequences for diagnostic tests and residuals.

## Setup

Before running, ensure:
- `bitcoin_daily_2016_2025.csv` is in the project root.
- Dependencies are installed: `pip install -r requirements.txt`.

The notebook is structured to mirror the `main_analysis.py` script but with interactive cells for exploration.

## Project Structure Reference
```
midterm_project/
├── bitcoin_daily_2016_2025.csv  # Data file
├── code/                        # Source files (imported below)
├── results/                     # Outputs generated here
└── this_notebook.ipynb
```

## Step 1: Imports and Setup

Import necessary modules and functions from the project code.

In [1]:
import sys
import os
from pathlib import Path

# Add code directory to path if needed (adjust if running from different location)
code_dir = Path('./code')
if code_dir.exists():
    sys.path.append(str(code_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict

# Project-specific imports
from data_loader import (
    load_financial_data, 
    prepare_baseline_data, 
    prepare_shuffled_data
)
from models import (
    fit_arima_baseline,
    fit_arima_shuffled,
    generate_forecasts_arima
)
from analysis import generate_comprehensive_comparison
from visualization import (
    create_output_dir,
    plot_forecast_comparison,
    plot_error_metrics_comparison,
    plot_parameter_comparison,
    plot_diagnostic_comparison,
    plot_acf_comparison,
    plot_residual_comparison
)

# Warnings and plotting setup
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

# Create results directory
output_dir = create_output_dir('results/figures')
results_dir = Path('results/tables')
results_dir.mkdir(parents=True, exist_ok=True)

print('Setup complete!')

ModuleNotFoundError: No module named 'data_loader'

## Step 2: Data Loading and Exploration

Load the Bitcoin daily data and explore its structure.

In [ ]:
# Load data (adjust path if needed)
csv_path = 'C:\Users\b_nko\Downloads\course_files_export\bitcoin_daily_2016_2025.csv'  # Update if in different location
data = load_financial_data(csv_path)

# Explore data
print(f'Data shape: {data.shape}')
print(f'Date range: {data["Date"].min()} to {data["Date"].max()}')
display(data.head())
display(data.describe())

# Quick plot of returns
plt.figure(figsize=(12, 4))
plt.plot(data['Date'], data['Returns'])
plt.title('Bitcoin Daily Returns (2016-2025)')
plt.xlabel('Date')
plt.ylabel('Returns (%)')
plt.grid(True, alpha=0.3)
plt.show()

## Step 3: Data Preparation

Prepare baseline (ordered) and shuffled datasets with lag features and train/test splits.

In [ ]:
# Baseline data (unshuffled)
baseline_data = prepare_baseline_data(
    data, 
    target_col='Returns',
    max_lag=5,
    train_split=0.8
)
print('Baseline data prepared.')
print(f'Train shape: {baseline_data["train"].shape}, Test shape: {baseline_data["test"].shape}')
display(baseline_data['train'].head())

# Shuffled data
shuffled_data = prepare_shuffled_data(
    data,
    target_col='Returns',
    max_lag=5,
    train_split=0.8,
    random_seed=42
)
print('\nShuffled data prepared.')
print(f'Train shape: {shuffled_data["train"].shape}, Test shape: {shuffled_data["test"].shape}')
display(shuffled_data['train'].head())

## Step 4: Model Fitting

Fit ARIMA models to both datasets using AIC-based order selection.

In [ ]:
# Fit baseline ARIMA
baseline_model = fit_arima_baseline(
    baseline_data['train'],
    target_col='Returns',
    max_p=3,
    max_d=2,
    max_q=3
)
print(f'Baseline model: ARIMA{baseline_model["order"]} (AIC: {baseline_model["aic"]:.2f})')

# Fit shuffled ARIMA
shuffled_model = fit_arima_shuffled(
    shuffled_data['train'],
    target_col='Returns',
    max_p=3,
    max_d=2,
    max_q=3
)
print(f'Shuffled model: ARIMA{shuffled_model["order"]} (AIC: {shuffled_model["aic"]:.2f})')

## Step 5: Forecasting

Generate out-of-sample forecasts for evaluation.

In [ ]:
# Generate forecasts
baseline_forecasts = generate_forecasts_arima(
    baseline_model,
    baseline_data['test'],
    target_col='Returns',
    horizon=len(baseline_data['test'])
)

shuffled_forecasts = generate_forecasts_arima(
    shuffled_model,
    shuffled_data['test'],
    target_col='Returns',
    horizon=len(shuffled_data['test'])
)

print(f'Baseline forecasts shape: {baseline_forecasts.shape}')
print(f'Shuffled forecasts shape: {shuffled_forecasts.shape}')

## Step 6: Comprehensive Comparison

Run the full comparison across estimation bias, prediction accuracy, and inference validity.

In [ ]:
# Generate comparisons
comparison_results = generate_comprehensive_comparison(
    baseline_model,
    shuffled_model,
    baseline_data,
    shuffled_data,
    model_type='arima'
)

# Display key tables
print('=== ESTIMATION BIASNESS ===')
display(comparison_results['estimation_bias'])

print('\n=== PREDICTION ACCURACY ===')
display(comparison_results['prediction_accuracy'])

print('\n=== INFERENCE VALIDITY ===')
display(comparison_results['inference_validity'])

## Step 7: Visualizations

Generate and display/save all comparison plots.

In [ ]:
# Forecast comparison
plot_forecast_comparison(
    baseline_data['test']['Returns'],
    baseline_forecasts,
    shuffled_data['test']['Returns'],
    shuffled_forecasts,
    output_dir
)

# Error metrics
plot_error_metrics_comparison(
    comparison_results['prediction_accuracy'],
    output_dir
)

# Parameter comparison
plot_parameter_comparison(
    comparison_results['estimation_bias'],
    output_dir,
    top_n=15
)

# Diagnostic comparison
plot_diagnostic_comparison(
    comparison_results['inference_validity'],
    output_dir
)

# ACF comparison
plot_acf_comparison(
    baseline_data['train']['Returns'],
    shuffled_data['train']['Returns'],
    output_dir,
    max_lags=40
)

# Residual comparison
baseline_residuals = baseline_model['model'].resid.values
shuffled_residuals = shuffled_model['model'].resid.values
plot_residual_comparison(
    baseline_residuals,
    shuffled_residuals,
    output_dir
)

print('All visualizations saved to results/figures/')

## Step 8: Save Results Tables

Export comparison tables as CSVs for reporting.

In [ ]:
# Save tables
comparison_results['estimation_bias'].to_csv(results_dir / 'estimation_bias_comparison.csv', index=False)
comparison_results['prediction_accuracy'].to_csv(results_dir / 'prediction_accuracy_comparison.csv', index=False)
comparison_results['inference_validity'].to_csv(results_dir / 'inference_validity_comparison.csv', index=False)

print('Tables saved to results/tables/')

# Quick preview of saved files
for file in results_dir.glob('*.csv'):
    print(f'- {file.name}: {len(pd.read_csv(file))} rows')

## Summary and Key Findings

### Expected Insights (Based on Methodology)

| Dimension              | Baseline (Ordered) Expectation | Shuffled Expectation | Interpretation |
|------------------------|--------------------------------|----------------------|----------------|
| **Estimation Bias**   | Accurate AR/MA coefficients   | Biased toward zero  | Shuffling disrupts temporal dependencies, leading to underestimated dynamics. |
| **Prediction Accuracy**| Lower RMSE/MAE/MAPE           | Higher errors       | Loss of sequential patterns reduces forecast reliability. |
| **Inference Validity**| Better residual diagnostics   | Violated assumptions| Disrupted order invalidates time-series-specific tests (e.g., autocorrelation). |

Review the generated tables and figures for actual results. Use these in your presentation template (`presentation/presentation_template.md`).

### Next Steps
1. Interpret results: Focus on % changes in parameters and metrics.
2. Update presentation: Insert tables/figures into slides.
3. Practice: 15-min talk + 5-min Q&A.

For troubleshooting, refer to `QUICK_START.md`. Good luck!